# UniAccurate — Python quickstart

`uniaccurate` is a Cython extension over the UniAccurate C ABI, shipped as a
self-contained wheel: the native library travels inside the package, so
installing it needs neither Nim nor a compiler.

```
pip install uniaccurate
```

CI executes this notebook against the wheel the release actually publishes, so
an output below that stops matching fails the build.

## The API

In [1]:
import uniaccurate

uniaccurate.version(), uniaccurate.__version__

('0.0.1', '0.0.1')

`two_sum` is an error-free transformation: it returns the rounded
sum `s = fl(a + b)` and the exact rounding error `e`, with `a + b == s + e` in
real arithmetic.

In [2]:
uniaccurate.two_sum(1.0, 2.0)

(3.0, 0.0)

When the addend is too small to affect the sum, it is not lost —
it shows up as `e`. That recovered error is what compensated summation
threads through a long running sum.

In [3]:
uniaccurate.two_sum(1.0, 2e16)

(2e+16, 1.0)

Non-finite input is not a contract violation: `s` follows IEEE
arithmetic and the error reads `NaN`.

In [4]:
import math
s, e = uniaccurate.two_sum(float("inf"), 1.0)
math.isinf(s), math.isnan(e)

(True, True)

A non-numeric argument is a type error, not a coercion.

In [5]:
try:
    uniaccurate.two_sum("x", 2.0)
except TypeError as exc:
    print("TypeError:", exc)

TypeError: a must be a number, got str


## The C ABI underneath

The same entry point is reachable from anything that speaks C. There the
contract is expressed without raising — an exception must never unwind across
an ABI boundary:

```c
double s, e;
ua_two_sum(1.0, 2e16, &s, &e);   /* s = 2e16, e = 1.0 */
```

See `include/UniAccurate.h`, and the book for the full picture.